In [0]:
%sql
-- Verify the final Users Silver table and its ADLS location

DESCRIBE DETAIL adbdevbankproject.silver.users_data;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,3269b844-896e-4ad2-98bc-878fa9a3d55e,adbdevbankproject.silver.users_data,null,abfss://silver@stgdevbankproject.dfs.core.windows.net/users_data,2026-08-21T13:47:31.923Z,2026-08-21T13:47:32.000Z,List(),List(),1,68433,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Rename the External Users Silver table to the original table name

ALTER TABLE adbdevbankproject.silver.users_data_external
RENAME TO adbdevbankproject.silver.users_data;

In [0]:
%sql
-- Drop the original Managed Users Silver table after validation

DROP TABLE adbdevbankproject.silver.users_data;

In [0]:
%sql
-- Compare row counts between the original and External Users Silver tables

SELECT 
    'Original Silver Table' AS table_name,
    COUNT(*) AS row_count
FROM adbdevbankproject.silver.users_data

UNION ALL

SELECT 
    'External Silver Table' AS table_name,
    COUNT(*) AS row_count
FROM adbdevbankproject.silver.users_data_external;

table_name,row_count
Original Silver Table,2000
External Silver Table,2000


In [0]:
%sql
-- Verify the External Users Silver table location

DESCRIBE DETAIL adbdevbankproject.silver.users_data_external;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,3269b844-896e-4ad2-98bc-878fa9a3d55e,adbdevbankproject.silver.users_data_external,null,abfss://silver@stgdevbankproject.dfs.core.windows.net/users_data,2026-08-21T13:47:31.923Z,2026-08-21T13:47:32.000Z,List(),List(),1,68433,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Create the external Users Silver table in the Silver ADLS container

CREATE TABLE adbdevbankproject.silver.users_data_external
USING DELTA
LOCATION 'abfss://silver@stgdevbankproject.dfs.core.windows.net/users_data'
AS
SELECT *
FROM adbdevbankproject.silver.users_data;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify the Users Silver schema

DESCRIBE adbdevbankproject.silver.users_data;

col_name,data_type,comment
id,int,null
current_age,int,null
retirement_age,int,null
birth_year,int,null
birth_month,int,null
gender,string,null
latitude,double,null
longitude,double,null
per_capita_income,"decimal(12,2)",null
yearly_income,"decimal(12,2)",null


In [0]:
%sql
-- Verify the cleaned Users Silver data

SELECT *
FROM adbdevbankproject.silver.users_data
LIMIT 10;

id,current_age,retirement_age,birth_year,birth_month,gender,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,34.15,-117.76,29278.00,59696.00,127613.00,787,5
1746,53,68,1966,12,Female,40.76,-73.74,37891.00,77254.00,191349.00,701,5
1718,81,67,1938,11,Female,34.02,-117.89,22681.00,33483.00,196.00,698,5
708,63,63,1957,1,Female,40.71,-73.99,163145.00,249925.00,202328.00,722,4
1164,43,70,1976,9,Male,37.76,-122.44,53797.00,109687.00,183855.00,675,1
68,42,70,1977,10,Male,41.55,-90.6,20599.00,41997.00,0.00,704,3
1075,36,67,1983,12,Female,38.22,-85.74,25258.00,51500.00,102286.00,672,3
1711,26,67,1993,12,Male,45.51,-122.64,26790.00,54623.00,114711.00,728,1
1116,81,66,1938,7,Female,40.32,-75.32,26273.00,42509.00,2895.00,755,5
1752,34,60,1986,1,Female,29.97,-92.12,18730.00,38190.00,81262.00,810,1


In [0]:
%sql
-- Create the cleaned Users Silver table

CREATE OR REPLACE TABLE adbdevbankproject.silver.users_data
AS
SELECT
    id,
    current_age,
    retirement_age,
    birth_year,
    birth_month,
    gender,
    latitude,
    longitude,
    CAST(REPLACE(per_capita_income, '$', '') AS DECIMAL(12,2)) AS per_capita_income,
    CAST(REPLACE(yearly_income, '$', '') AS DECIMAL(12,2)) AS yearly_income,
    CAST(REPLACE(total_debt, '$', '') AS DECIMAL(12,2)) AS total_debt,
    credit_score,
    num_credit_cards
FROM adbdevbankproject.bronze.users;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Review the Users table schema and data types

DESCRIBE adbdevbankproject.bronze.users;

col_name,data_type,comment
id,int,null
current_age,int,null
retirement_age,int,null
birth_year,int,null
birth_month,int,null
gender,string,null
address,string,null
latitude,double,null
longitude,double,null
per_capita_income,string,null


In [0]:
%sql
-- Review the Users Bronze table before applying Silver transformations

SELECT *
FROM adbdevbankproject.bronze.users
LIMIT 10;

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,$20599,$41997,$0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,$26273,$42509,$2895,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,$18730,$38190,$81262,810,1
